# ONNX Export — Mualem Wav2Vec2-BERT Multilevel CTC

Exports the model to ONNX. Supports QAT (INT8 dequantized to FP32) and plain FP32 checkpoints.

In [ ]:
!pip install -q torch transformers safetensors onnx onnxruntime-gpu gdown numpy onnxscript datasets soundfile

In [ ]:
USE_FOLDER_LINK = False
GDRIVE_LINKS = {
    "config.json":              "https://drive.google.com/file/d/1BlMtr4ole4Z3cY4eyrrnJN5pZgv2GJDY/view?usp=sharing",
    "model_best.safetensors":   "https://drive.google.com/file/d/12VaJWyA9AgXrETu1JkDttNcWBakVsb1Q/view?usp=sharing",
    "vocab.json":               "https://drive.google.com/file/d/1vjDd4XGRcLsfqXqOZht3_-7rfsoaQ7dg/view?usp=sharing",
    "preprocessor_config.json": "https://drive.google.com/file/d/12QcW734UG1HekXTOiRWikfQo3i1opTzP/view?usp=sharing",
}
GDRIVE_FOLDER_LINK = "PASTE_YOUR_GOOGLE_DRIVE_FOLDER_LINK_HERE"
WEIGHTS_FILENAME = "model_best.safetensors"
OPSET_VERSION = 17
EXPORT_FP16 = False

# True  -> QAT (INT8) checkpoint, dequantized to FP32 for export
# False -> plain FP32 checkpoint
USE_QAT_CHECKPOINT = False

APPLY_ONNX_QUANTIZATION = True
# "dynamic" -> weights INT8, activations FP32, no calibration needed
# "static"  -> weights + activations INT8, requires calibration audio
# "both"    -> runs both modes
QUANTIZATION_MODE = "both"

HF_CALIB_DATASET  = "obadx/muaalem-annotated-v3"
HF_CALIB_SUBSET   = "moshaf_0.0"
NUM_CALIB_SAMPLES = 8  # keep low (8-16) to avoid OOM

In [ ]:
import os, sys, json, time, gc, re, shutil
from pathlib import Path
import gdown

WORK_DIR   = Path("/kaggle/working")
MODEL_DIR  = WORK_DIR / "model_download"
EXPORT_DIR = WORK_DIR / "onnx_export"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def extract_gdrive_id(url):
    for pat in [r'/d/([a-zA-Z0-9_-]+)', r'[?&]id=([a-zA-Z0-9_-]+)', r'/folders/([a-zA-Z0-9_-]+)']:
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return url.strip()

if USE_FOLDER_LINK:
    gdown.download_folder(id=extract_gdrive_id(GDRIVE_FOLDER_LINK), output=str(MODEL_DIR), quiet=False)
else:
    for fname, link in GDRIVE_LINKS.items():
        dest = MODEL_DIR / fname
        if dest.exists():
            print(f"  {fname} (cached)")
            continue
        print(f"  downloading {fname}...")
        gdown.download(f"https://drive.google.com/uc?id={extract_gdrive_id(link)}", str(dest), quiet=False)

for f in ["config.json", WEIGHTS_FILENAME, "vocab.json", "preprocessor_config.json"]:
    p = MODEL_DIR / f
    if not p.exists():
        matches = list(MODEL_DIR.rglob(f))
        if matches:
            shutil.copy2(str(matches[0]), str(p))
        else:
            raise FileNotFoundError(f"Missing: {f}")

In [ ]:
import torch
import torch.nn as nn
from safetensors.torch import load_file
from transformers import Wav2Vec2BertConfig, Wav2Vec2BertModel

HEAD_DIM = 64

class Wav2Vec2BertForMultilevelCTC(nn.Module):
    def __init__(self, config, vocab_sizes):
        super().__init__()
        self.wav2vec2_bert = Wav2Vec2BertModel(config)
        self.dropout = nn.Dropout(config.final_dropout)
        self.ctc_heads = nn.ModuleDict({
            name: nn.Linear(config.hidden_size, vs, bias=True)
            for name, vs in vocab_sizes.items()
        })

    def forward(self, input_features, attention_mask=None):
        out = self.wav2vec2_bert(input_features=input_features, attention_mask=attention_mask)
        h = self.dropout(out.last_hidden_state)
        return {name: head(h) for name, head in self.ctc_heads.items()}

with open(MODEL_DIR / "config.json", encoding="utf-8") as f:
    cfg = json.load(f)
level_to_vocab_size = cfg.pop("level_to_vocab_size")
for k in ("level_to_loss_weight", "architectures", "model_type", "transformers_version"):
    cfg.pop(k, None)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Wav2Vec2BertForMultilevelCTC(Wav2Vec2BertConfig(**cfg), level_to_vocab_size)

if USE_QAT_CHECKPOINT:
    ckpt = load_file(str(MODEL_DIR / WEIGHTS_FILENAME))
    remapped = {
        (k.replace("level_to_lm_head.", "ctc_heads.") if k.startswith("level_to_lm_head.") else k): v
        for k, v in ckpt.items()
    }
    del ckpt

    # Phase 1: rebuild all nn.Linear layers, dequantizing INT8 where needed
    rebuilt_deq, rebuilt_fp, matched, skipped_lin = 0, 0, 0, 0
    for module_path, module in list(model.named_modules()):
        if not isinstance(module, nn.Linear):
            continue

        w_key       = f"{module_path}.weight"
        w_int8_key  = f"{module_path}.weight_int8"
        w_scale_key = f"{module_path}.weight_scale"
        w_zp_key    = f"{module_path}.weight_zp"
        b_key       = f"{module_path}.bias"

        if w_key in remapped:
            weight_fp = remapped[w_key]
            source = "fp32"
        elif w_int8_key in remapped:
            w_int8  = remapped[w_int8_key]
            w_scale = remapped[w_scale_key]
            w_zp    = remapped[w_zp_key]
            weight_fp = (w_int8.float() - w_zp) * w_scale
            source = "int8"
        else:
            skipped_lin += 1
            continue

        bias_fp = remapped.get(b_key)
        new_linear = nn.Linear(weight_fp.shape[1], weight_fp.shape[0], bias=(bias_fp is not None))
        with torch.no_grad():
            new_linear.weight.copy_(weight_fp)
            if bias_fp is not None:
                new_linear.bias.copy_(bias_fp)

        parts = module_path.split(".")
        parent = model
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], new_linear)

        if source == "int8":
            rebuilt_deq += 1
        elif module.weight.shape != weight_fp.shape:
            rebuilt_fp += 1
        else:
            matched += 1

    print(f"Phase 1: dequantized={rebuilt_deq}, rebuilt={rebuilt_fp}, matched={matched}, skipped={skipped_lin}")

    linear_handled_prefixes = {
        path for path, mod in model.named_modules() if isinstance(mod, nn.Linear)
    }

    # Phase 2: load remaining params (LayerNorm, Conv, etc.)
    skip_suffixes = (".weight", ".bias", ".weight_int8", ".weight_scale",
                     ".weight_zp", ".act_scale", ".act_zp", ".act_qmin", ".act_qmax")
    loaded, skipped_p = 0, 0
    for name, tensor in remapped.items():
        prefix = name
        for sfx in skip_suffixes:
            if name.endswith(sfx):
                prefix = name[:-len(sfx)]
                break
        if prefix in linear_handled_prefixes:
            continue

        parts = name.split(".")
        obj = model
        try:
            for p in parts[:-1]:
                obj = getattr(obj, p)
        except AttributeError:
            skipped_p += 1
            continue
        attr = parts[-1]

        if attr in obj._parameters and obj._parameters[attr] is not None:
            if obj._parameters[attr].shape == tensor.shape:
                with torch.no_grad():
                    obj._parameters[attr].data.copy_(tensor)
                loaded += 1
            else:
                print(f"  shape mismatch: {name} model={list(obj._parameters[attr].shape)} ckpt={list(tensor.shape)}")
                skipped_p += 1
        elif attr in obj._buffers:
            if obj._buffers[attr] is not None and obj._buffers[attr].shape == tensor.shape:
                with torch.no_grad():
                    obj._buffers[attr].copy_(tensor)
                loaded += 1
            else:
                skipped_p += 1
        else:
            skipped_p += 1

    print(f"Phase 2: loaded={loaded}, skipped={skipped_p}")
    del remapped
    gc.collect()

else:
    ckpt = load_file(str(MODEL_DIR / WEIGHTS_FILENAME))
    remapped = {
        (k.replace("level_to_lm_head.", "ctc_heads.") if k.startswith("level_to_lm_head.") else k): v
        for k, v in ckpt.items()
    }
    del ckpt

    missing, unexpected = model.load_state_dict(remapped, strict=False)
    print(f"Missing keys: {len(missing)}, unexpected keys: {len(unexpected)}")
    del remapped
    gc.collect()

# Phase 3: patch attention head counts to match pruned weights
for layer in model.wav2vec2_bert.encoder.layers:
    a = layer.self_attn
    actual_heads = a.linear_q.weight.shape[0] // HEAD_DIM
    a.num_heads = actual_heads
    a.head_dim  = HEAD_DIM
    if hasattr(a, "head_size"):
        a.head_size = HEAD_DIM
if hasattr(model.wav2vec2_bert, "adapter") and model.wav2vec2_bert.adapter:
    for al in model.wav2vec2_bert.adapter.layers:
        if hasattr(al, "self_attn"):
            al.self_attn.num_heads = al.self_attn.linear_q.weight.shape[0] // HEAD_DIM
            al.self_attn.head_dim = HEAD_DIM

model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded ({device}) — {n_params:.1f}M params")
print(f"CTC heads: {list(level_to_vocab_size.keys())}")

layer0 = model.wav2vec2_bert.encoder.layers[0]
print(f"Layer 0: heads={layer0.self_attn.num_heads}, "
      f"q={list(layer0.self_attn.linear_q.weight.shape)}, "
      f"ffn1={list(layer0.ffn1.intermediate_dense.weight.shape)}, "
      f"ffn2={list(layer0.ffn2.intermediate_dense.weight.shape)}")

In [ ]:
import numpy as np
from transformers import SeamlessM4TFeatureExtractor

class OnnxWrapper(nn.Module):
    def __init__(self, model, level_names):
        super().__init__()
        self.model = model
        self.level_names = sorted(level_names)

    def forward(self, input_features):
        logits = self.model(input_features)
        return tuple(logits[name] for name in self.level_names)

level_names = sorted(level_to_vocab_size.keys())
wrapper = OnnxWrapper(model, level_names)
wrapper.eval()

fe = SeamlessM4TFeatureExtractor.from_pretrained(str(MODEL_DIR))
dummy_audio = np.random.randn(16000).astype(np.float32)
dummy_feats = fe([dummy_audio], sampling_rate=16000, return_tensors="pt", padding=True)
dummy_input = dummy_feats["input_features"].to(device)
FEATURE_DIM = dummy_input.shape[-1]

with torch.no_grad():
    test_out = wrapper(dummy_input)
for name, t in zip(level_names, test_out):
    print(f"  {name:30s} -> {list(t.shape)}")

In [ ]:
import onnx

onnx_path = EXPORT_DIR / "mualem_multilevel_ctc.onnx"
dynamic_axes = {"input_features": {0: "batch_size", 1: "seq_len"}}
for name in level_names:
    dynamic_axes[name] = {0: "batch_size", 1: "time_steps"}

t0 = time.time()
torch.onnx.export(
    wrapper, (dummy_input,), str(onnx_path),
    input_names=["input_features"], output_names=level_names,
    dynamic_axes=dynamic_axes, opset_version=OPSET_VERSION,
    do_constant_folding=True, export_params=True,
    dynamo=False,  # legacy TorchScript exporter for stable dynamic_axes + opset 17
)
print(f"Exported in {time.time()-t0:.1f}s")

# Inline external data back into a single file if PyTorch split it
data_file = Path(str(onnx_path) + ".data")
if data_file.exists():
    from onnx.external_data_helper import load_external_data_for_model
    _m = onnx.load(str(onnx_path), load_external_data=False)
    load_external_data_for_model(_m, str(EXPORT_DIR))
    onnx.save_model(_m, str(onnx_path), save_as_external_data=False)
    data_file.unlink()
    del _m
    gc.collect()

print(f"ONNX: {onnx_path.stat().st_size/1e6:.1f} MB")

In [ ]:
import io as _io
import soundfile as _sf
from datasets import Audio, load_dataset
from onnxruntime.quantization import (
    quantize_dynamic, quantize_static, QuantType, CalibrationDataReader,
)

q_dyn_path    = EXPORT_DIR / "mualem_multilevel_ctc_int8_dynamic.onnx"
q_static_path = EXPORT_DIR / "mualem_multilevel_ctc_int8_static.onnx"

if not APPLY_ONNX_QUANTIZATION:
    print("APPLY_ONNX_QUANTIZATION=False, skipping PTQ")
else:
    sz_fp = onnx_path.stat().st_size / 1e6

    if QUANTIZATION_MODE in ("dynamic", "both"):
        if q_dyn_path.exists():
            print(f"INT8 dynamic already exists: {q_dyn_path.stat().st_size/1e6:.1f} MB (skipping)")
        else:
            print("Running dynamic quantization...")
            quantize_dynamic(
                str(onnx_path), str(q_dyn_path),
                weight_type=QuantType.QInt8,
                extra_options={"MatMulConstBOnly": True},
            )
            sz_dyn = q_dyn_path.stat().st_size / 1e6
            print(f"  FP32: {sz_fp:.1f} MB  ->  INT8 dynamic: {sz_dyn:.1f} MB ({100*sz_dyn/sz_fp:.0f}%)")

    if QUANTIZATION_MODE in ("static", "both"):
        print(f"Running static quantization ({HF_CALIB_DATASET}/{HF_CALIB_SUBSET}, {NUM_CALIB_SAMPLES} samples)...")
        torch.cuda.empty_cache()
        gc.collect()

        class HFCalibReader(CalibrationDataReader):
            def __init__(self):
                ds = load_dataset(HF_CALIB_DATASET, HF_CALIB_SUBSET, split="train", streaming=True)
                ds = ds.cast_column("audio", Audio(decode=False))
                self._data = []
                self._idx = 0
                for sample in ds.take(NUM_CALIB_SAMPLES):
                    buf = _io.BytesIO(sample["audio"]["bytes"])
                    wav, sr = _sf.read(buf, dtype="float32")
                    if wav.ndim > 1:
                        wav = wav.mean(axis=1)
                    feats = fe([wav], sampling_rate=16000, return_tensors="np", padding=True)
                    self._data.append({"input_features": feats["input_features"]})
                print(f"  Loaded {len(self._data)} calibration samples")

            def get_next(self):
                if self._idx >= len(self._data):
                    return None
                d = self._data[self._idx]
                self._idx += 1
                return d

        quantize_static(
            str(onnx_path), str(q_static_path),
            HFCalibReader(),
            weight_type=QuantType.QInt8,
            activation_type=QuantType.QInt8,
            per_channel=False,
            optimize_model=False,
        )
        sz_st = q_static_path.stat().st_size / 1e6
        print(f"  FP32: {sz_fp:.1f} MB  ->  INT8 static: {sz_st:.1f} MB ({100*sz_st/sz_fp:.0f}%)")

    print("PTQ complete")

In [ ]:
print("Validating ONNX graph...")
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
for inp in onnx_model.graph.input:
    dims = [d.dim_param or d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f"  in  {inp.name}: {dims}")
for out in onnx_model.graph.output:
    dims = [d.dim_param or d.dim_value for d in out.type.tensor_type.shape.dim]
    print(f"  out {out.name}: {dims}")
del onnx_model

import onnxruntime as ort
try:
    sess = ort.InferenceSession(str(onnx_path), providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
except Exception:
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
print(f"Provider: {sess.get_providers()[0]}")

ok = True
for dur in [0.5, 1.0, 2.0, 5.0]:
    a = np.random.randn(int(16000 * dur)).astype(np.float32)
    f = fe([a], sampling_rate=16000, return_tensors="pt", padding=True)
    inp = f["input_features"].to(device)
    with torch.no_grad():
        pt = wrapper(inp)
    ort_o = sess.run(None, {"input_features": inp.cpu().numpy()})
    diffs = [np.abs(pt[i].cpu().numpy() - ort_o[i]).max() for i in range(len(level_names))]
    md = max(diffs)
    status = "OK" if md < 1e-3 else ("warn" if md < 1e-2 else "FAIL")
    if md >= 1e-2:
        ok = False
    print(f"  {dur:.1f}s T={inp.shape[1]:>4d}  diff={md:.6f}  {status}")
print("All passed." if ok else "Some diffs exceeded threshold.")

In [ ]:
for f in ["vocab.json", "preprocessor_config.json", "config.json"]:
    src = MODEL_DIR / f
    if src.exists():
        shutil.copy2(str(src), str(EXPORT_DIR / f))

meta = {
    "model": "Wav2Vec2BertForMultilevelCTC (QAT-dequantized)",
    "source": WEIGHTS_FILENAME,
    "opset": OPSET_VERSION,
    "input": f"(B, T, {FEATURE_DIM})",
    "outputs": level_names,
    "vocab_sizes": level_to_vocab_size,
    "params_M": round(n_params, 2),
    "note": "INT8 QAT weights dequantized to FP32 for ONNX export",
}
(EXPORT_DIR / "onnx_metadata.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))

print("ONNX Export Summary")
print("-" * 40)
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"  {f.name:45s} {f.stat().st_size/1e6:>8.1f} MB")
print(f"\n{n_params:.1f}M params | Input: (B, T, {FEATURE_DIM}) | {len(level_names)} levels")

In [ ]:
with open(EXPORT_DIR / "vocab.json", encoding="utf-8") as f:
    vocab = json.load(f)

id2tok = {}
for lv in level_names:
    id2tok[lv] = {token_id: token for token, token_id in vocab[lv].items()}

def ctc_decode(logits, lv):
    ids = logits.argmax(-1).flatten().tolist()
    out, prev = [], None
    for i in ids:
        if i != prev:
            out.append(i)
        prev = i
    return "".join(id2tok[lv].get(i, f"[{i}]") for i in out if i != 0)

da = np.random.randn(16000).astype(np.float32)
df = fe([da], sampling_rate=16000, return_tensors="np", padding=True)
ort_o = sess.run(None, {"input_features": df["input_features"]})
for name, logits in zip(level_names, ort_o):
    decoded = ctc_decode(logits, name)
    suffix = "..." if len(decoded) > 80 else ""
    print(f"  {name:30s} -> {decoded[:80]}{suffix}")

---
## Usage
```python
import onnxruntime as ort
from transformers import SeamlessM4TFeatureExtractor
import soundfile as sf

sess = ort.InferenceSession("mualem_multilevel_ctc.onnx")
fe = SeamlessM4TFeatureExtractor.from_pretrained("./onnx_export/")
audio, sr = sf.read("audio.wav", dtype="float32")
feats = fe([audio], sampling_rate=16000, return_tensors="np", padding=True)
outputs = sess.run(None, {"input_features": feats["input_features"]})
# outputs[i] corresponds to level_names[i]; apply argmax + CTC decode
```